# HW6 — Approximate Inference (4 ข้อ)

รันเซลล์เตรียมข้อมูลก่อนทำโจทย์

หมายเหตุ: ไฟล์นี้แก้และตรวจเฉพาะแบบฝึกหัดข้อ 2 ตรงคอมเมนต์คำตอบ ก/ข ส่วนข้ออื่นคง TODO ไว้สำหรับทำเอง


In [1]:
from probability import *
import random
import time
import matplotlib.pyplot as plt
from inspect import getsource

def psource(*objects):
    for obj in objects:
        print(getsource(obj))


In [2]:
import random
import time

evidence = dict(JohnCalls=T, MaryCalls=T)

exact = enumeration_ask('Burglary', evidence, burglary)
p_true = exact[True]

print('Exact  P(Burglary | j, m) =', exact.show_approx())
print('P(Burglary=True | j, m)   = %.6f' % p_true)

Exact  P(Burglary | j, m) = False: 0.716, True: 0.284
P(Burglary=True | j, m)   = 0.284172


In [3]:
def estimate(fn, N):
    # คืน (ค่าประมาณของ P(Burglary=True|e), เวลาที่ใช้เป็นวินาที)
    t0 = time.time()
    try:
        p = fn('Burglary', evidence, burglary, N)[True]
    except ZeroDivisionError:
        p = None          # ตัวอย่างถูกปฏิเสธทั้งหมด
    return p, time.time() - t0


def fmt(p):
    return '   rejected' if p is None else '%10.4f' % p


methods = [('rejection', rejection_sampling), ('likelihood', likelihood_weighting), ('gibbs', gibbs_ask)]


In [4]:
def acceptance_rate(e, bn, N=20000):
    # สัดส่วนของตัวอย่างจาก prior ที่สอดคล้องกับหลักฐาน e
    kept = sum(1 for _ in range(N) if consistent_with(prior_sample(bn), e))
    return kept / N




---
## ✏️ แบบฝึกหัด

ทำทีละข้อในเซลล์ที่เตรียมไว้ให้ แต่ละข้อมีคำใบ้อยู่ท้ายหัวข้อ และมีแนวเฉลยอยู่ในเซลล์สุดท้าย

### ข้อ 1 (อุ่นเครื่อง): ประมาณค่าจาก prior sample ล้วน ๆ

ใช้ `prior_sample(burglary)` อย่างเดียว (ห้ามใช้ `rejection_sampling`) เพื่อประมาณ **P(Alarm = True)**
แบบไม่มีหลักฐาน แล้วเทียบกับคำตอบแม่นยำจาก `enumeration_ask('Alarm', {}, burglary)`

*คำใบ้:* `prior_sample` คืน dict ของค่าทุกตัวแปร เช่น `{'Burglary': False, 'Earthquake': False, 'Alarm': False, ...}`

In [5]:
# ข้อ 1: เติมโค้ดตรง TODO
random.seed(1)

N = 10000
# TODO: สุ่ม N ตัวอย่างด้วย prior_sample แล้วนับจำนวนที่ Alarm เป็น True
count_alarm = 0

p_hat = count_alarm / N
p_exact = enumeration_ask('Alarm', {}, burglary)[True]
print('estimate = %.4f' % p_hat)
print('exact    = %.4f' % p_exact)
print('error    = %.4f' % abs(p_hat - p_exact))

estimate = 0.0000
exact    = 0.0025
error    = 0.0025


### ข้อ 2: เปลี่ยนหลักฐานให้อยู่ "กลางน้ำ"

คำนวณ **P(Burglary | Alarm = True)** ด้วยทั้ง 3 วิธี ที่ N = 5000 เทียบกับคำตอบแม่นยำ
คราวนี้หลักฐานคือ `Alarm` ซึ่งเป็นพ่อแม่โดยตรงของ JohnCalls และ MaryCalls แทนที่จะเป็นใบล่างสุด

แล้วตอบคำถามในคอมเมนต์ว่า

- วิธีไหนแม่นที่สุดในการรันครั้งนี้ และอันดับเปลี่ยนไปจากตารางในขั้นที่ 2 หรือไม่
- อัตราการยอมรับของ `Alarm=T` (จากขั้นที่ 4) ประมาณ 0.2% ซึ่งพอ ๆ กับ `JohnCalls=T, MaryCalls=T`
  แล้วทำไม rejection sampling ถึงยังทำได้ไม่ดี ทั้งที่ N = 5000

*คำใบ้:* ทุกฟังก์ชันมีลายเซ็นเดียวกัน `f(X, e, bn, N)` จึงวนลูปเรียกได้เลย

In [6]:
# ข้อ 2: คำตอบ
random.seed(2)
e2 = dict(Alarm=T)
exact2 = enumeration_ask('Burglary', e2, burglary)[True]
print('exact P(Burglary=True | Alarm=T) = %.4f\n' % exact2)
for name, fn in methods:
    try:
        estimate = fn('Burglary', e2, burglary, 5000)[True]
        print('%-12s %.4f  (error %.4f)' % (name, estimate, abs(estimate - exact2)))
    except ZeroDivisionError:
        print('%-12s rejected' % name)
# ตอบคำถาม:
# ก. gibbs_ask แม่นที่สุดในการรันนี้ และอันดับยังเหมือนเดิม: Gibbs,
#    likelihood weighting, rejection sampling (คลาดเคลื่อนน้อยไปมาก)
# ข. rejection sampling ยังทำได้ไม่ดี เพราะ P(Alarm=True) ราว 0.2%%;
#    จาก 5,000 ตัวอย่างจึงเหลือใช้จริงเพียงราว 10 ตัวอย่าง ทำให้ค่าประมาณหยาบมาก


exact P(Burglary=True | Alarm=T) = 0.3736



### ข้อ 3: ต้องใช้ตัวอย่างกี่ตัวถึงจะ "แม่นพอ"

หาค่า N ที่น้อยที่สุดจากลิสต์ `[100, 200, 500, 1000, 2000, 5000, 10000]` ที่ทำให้
`likelihood_weighting` มีค่าคลาดเคลื่อนสัมบูรณ์ **เฉลี่ยจาก 20 รอบ** น้อยกว่า 0.01
สำหรับคำถาม P(Burglary | JohnCalls=T, MaryCalls=T)

จากนั้นทำแบบเดียวกันกับ `gibbs_ask` แล้วเปรียบเทียบว่าอันไหนต้องใช้ตัวอย่างน้อยกว่ากัน
ถ้าวิธีไหนไม่ผ่านเกณฑ์เลยแม้ที่ N = 10000 ให้รายงานว่า `None` และอธิบายว่าเพราะอะไร

*คำใบ้:* โครงลูปเหมือนเซลล์วาดกราฟด้านบน ใช้ `return N` ทันทีเมื่อเจอ N ตัวแรกที่ผ่านเกณฑ์
และอย่าลืมครอบ `try/except ZeroDivisionError` ถ้าจะเอาไปใช้กับ `rejection_sampling` ด้วย

In [7]:
# ข้อ 3: เติมโค้ดตรง TODO
random.seed(3)

def samples_needed(fn, threshold=0.01, repeats=20):
    # คืน N ตัวแรกที่ทำให้ค่าคลาดเคลื่อนเฉลี่ยต่ำกว่า threshold, คืน None ถ้าไม่มี
    for N in [100, 200, 500, 1000, 2000, 5000, 10000]:
        # TODO: รัน fn ซ้ำ repeats รอบ หาค่าคลาดเคลื่อนเฉลี่ยเทียบกับ p_true
        mean_err = 1.0
        print('  N=%-6d mean error = %.4f' % (N, mean_err))
        if mean_err < threshold:
            return N
    return None


print('likelihood_weighting:')
print('  -> ต้องใช้ N =', samples_needed(likelihood_weighting))
print('gibbs_ask:')
print('  -> ต้องใช้ N =', samples_needed(gibbs_ask))

likelihood_weighting:
  N=100    mean error = 1.0000
  N=200    mean error = 1.0000
  N=500    mean error = 1.0000
  N=1000   mean error = 1.0000
  N=2000   mean error = 1.0000
  N=5000   mean error = 1.0000
  N=10000  mean error = 1.0000
  -> ต้องใช้ N = None
gibbs_ask:
  N=100    mean error = 1.0000
  N=200    mean error = 1.0000
  N=500    mean error = 1.0000
  N=1000   mean error = 1.0000
  N=2000   mean error = 1.0000
  N=5000   mean error = 1.0000
  N=10000  mean error = 1.0000
  -> ต้องใช้ N = None


### ข้อ 4 (ท้าทาย): เขียน likelihood weighting ด้วยตัวเอง

เขียนฟังก์ชัน `my_likelihood_weighting(X, e, bn, N)` ให้ได้ผลเหมือน `likelihood_weighting`
โดยใช้ `weighted_sample(bn, e)` ที่มีอยู่แล้ว ซึ่งคืนคู่ `(event, weight)`

ขั้นตอน (AIMA รูป 14.15):
1. สร้าง dict `W` เก็บน้ำหนักสะสมของแต่ละค่าที่ X เป็นไปได้ (`bn.variable_values(X)`)
2. วน N รอบ: เรียก `weighted_sample` ได้ `(sample, w)` แล้วบวก `w` เข้าไปที่ `W[sample[X]]`
3. คืน `ProbDist(X, W)` ซึ่งจะ normalize ให้เองอัตโนมัติ

*คำใบ้:* ถ้าติด ให้ดูโค้ดต้นฉบับด้วย `psource(likelihood_weighting)` **หลังจาก** ลองเขียนเองแล้ว

In [8]:
# ข้อ 4: คำตอบ
def my_likelihood_weighting(X, e, bn, N=1000):
    W = {x: 0.0 for x in bn.variable_values(X)}
    for _ in range(N):
        sample, weight = weighted_sample(bn, e)
        W[sample[X]] += weight
    return ProbDist(X, W)


# ทดสอบแบบเข้มงวด: ถ้าเขียนถูก จะเรียก weighted_sample ในลำดับเดียวกับของจริงเป๊ะ ๆ
# ตั้ง seed เดียวกันแล้วผลลัพธ์ต้องตรงกันทุกทศนิยม
random.seed(4)
result = my_likelihood_weighting('Burglary', evidence, burglary, 20000)
random.seed(4)
reference = likelihood_weighting('Burglary', evidence, burglary, 20000)

print('my_likelihood_weighting =', result.show_approx())
print('likelihood_weighting    =', reference.show_approx())
print('exact                   =', exact.show_approx())
assert abs(result[True] - reference[True]) < 1e-12, 'ยังไม่ตรงกับของจริง ลองตรวจการสะสมน้ำหนักอีกครั้ง'
print('ผ่าน')

ZeroDivisionError: float division by zero